In [ ]:
import os
from sqlalchemy import create_engine, text
import pandas as pd

def is_docker_environment() -> bool:
    """
    Detect if running inside a Docker/Jupyter container.
    """
    # 1) Standard Docker marker
    if os.path.exists("/.dockerenv"):
        return True

    # 2) Heuristic: Jupyter in container often sets this,
    #    and our compose uses PGHOST=timescaledb inside the container.
    if os.environ.get("JUPYTER_SERVER_ROOT"):
        return True

    return False

def get_default_pg_host() -> str:
    """
    Choose sensible default host based on environment:
    - Inside Docker/Jupyter container: talk to 'timescaledb' service
    - Local (Mac): talk to Docker via 'localhost'
    """
    return "timescaledb" if is_docker_environment() else "localhost"

# Respect explicit PGHOST if user sets it; otherwise use env-aware default
PGUSER = os.getenv("PGUSER", "postgres")
PGPASSWORD = os.getenv("PGPASSWORD", "postgres")
PGHOST = os.getenv("PGHOST", get_default_pg_host())
PGPORT = os.getenv("PGPORT", "5432")
PGDATABASE = os.getenv("PGDATABASE", "factor_db")

db_url = f"postgresql+psycopg2://{PGUSER}:{PGPASSWORD}@{PGHOST}:{PGPORT}/{PGDATABASE}"
engine = create_engine(db_url, future=True)

try:
    with engine.begin() as conn:
        conn.execute(text("SELECT 1;"))
    env = "Docker/Jupyter" if is_docker_environment() else "Local"
    print(f"✅ Connected to TimescaleDB/Postgres ({env})")
    print(f"   URL: {db_url}")
except Exception as e:
    print(f"❌ Failed to connect to TimescaleDB/Postgres: {e}")
    raise


In [ ]:
with engine.connect() as conn:
    char_df = pd.read_sql(
        text("SELECT * FROM jkp_characteristics_with_sector;"),
        conn,
)

In [ ]:
char_df.shape

In [ ]:
char_df = char_df.drop(['ticker'], axis=1)

In [ ]:
char_df.drop_duplicates(subset=['gvkey', 'eom'], keep='first', inplace=True)

In [ ]:
char_df.shape

In [ ]:
#verify unique mappings

dupe_mask = char_df.duplicated(subset=["eom", "gvkey"], keep=False)
# dupes = char_df[dupe_mask]
dupes = char_df[dupe_mask].sort_values(["conm", "eom"])
dupes

### check for irregularities or jumps in any characteristics (should be cleaned by not quite sure)

In [ ]:
firm_count = char_df.groupby("eom")["gvkey"].nunique()
firm_count.plot()

In [ ]:
#check for any rows that have over 50% null
null_counts = char_df.isna().sum(axis=1)
high_null_mask = null_counts > (0.5 * char_df.shape[1])
high_null_rows = char_df[high_null_mask & char_df['conm'].isna()]
high_null_rows

In [ ]:
char_df.columns

In [ ]:
# check for huge jumps in time series data
jump_rows = []

for key, group in char_df.groupby("gvkey"):
    group = group.sort_values("eom").copy()
    for col in char_df.columns:
        if col not in ["gvkey", "size_grp", "eom", "id", "conm", "sic", "sector",
                       "permno_x", "permno_y", "permco", "gsector", "ggroup",
                       "gind", "gsubind", "Sector", "IndustryGroup", "Industry",
                       "SubIndustry", "SubIndustryDescription"]:
            # do NOT forward-fill NaNs before computing pct_change
            group[f"{col}_pct_change"] = group[col].pct_change(fill_method=None)

    large_jumps = group[
        (group.filter(like="_pct_change").abs() > 50).any(axis=1)
    ]

    if not large_jumps.empty:
        jump_rows.append(large_jumps)

# Combine all large jumps for all gvkeys into a single DataFrame
if jump_rows:
    large_jumps_df = pd.concat(jump_rows, ignore_index=True)
else:
    large_jumps_df = pd.DataFrame()

large_jumps_df

In [ ]:
group_jumps = large_jumps_df.groupby("gvkey")

for gvkey, jump in group_jumps:
    print(jump)
    break
    

In [ ]:
df_003382 = char_df[char_df['gvkey'] == '003382'].sort_values('eom')
df_003382[['gvkey', 'eom', 'oaccruals_at', 'at_gr1', 'conm']]

Shutterstock had no recent stock actions at those times

In [ ]:
char_df[char_df['ret_exc'].abs() > 10][['ret_exc', 'conm']]
# large_jumps_df[large_jumps_df['ret_exc'].abs() > 1]

### Standardizing

In [ ]:
char_cs_zscore = char_df.copy()
for col in char_df.columns:
    if col not in ["gvkey", "size_grp", "eom", "id", "conm", "sic", "sector",
                   "permno_x", "permno_y", "permco", "gsector", "ggroup",
                   "gind", "gsubind", "Sector", "IndustryGroup", "Industry",
                   "SubIndustry", "SubIndustryDescription"]:
        char_cs_zscore[col] = char_df.groupby("eom")[col].transform(
            lambda x: (x - x.mean()) / x.std()
        )

In [ ]:
char_cs_zscore

In [ ]:
nan_ratio = sum(char_cs_zscore.isna().any(axis=1)) / char_cs_zscore.shape[0]
nan_ratio

In [ ]:
# Define the columns of interest
fama_french_5_columns = [
    'be_me', 'ni_me', 'ocf_me', 'market_equity', 'ret_12_1', 
    'gp_at', 'ni_at', 'at_gr1', 'oaccruals_at', 'ret_exc', 'beta_60m'
]

missing = char_cs_zscore[fama_french_5_columns].isna().any(axis=1).mean()

missing

In [ ]:
char_cs_zscore = char_cs_zscore.sort_values(['gvkey', 'eom'])

ffill_cols = [
    'be_me', 'ni_me', 'ocf_me', 'market_equity', 'ret_12_1',
    'gp_at', 'ni_at', 'at_gr1', 'oaccruals_at'
]

char_cs_zscore_clean = char_cs_zscore.copy()

char_cs_zscore_clean[ffill_cols] = (
    char_cs_zscore_clean
    .groupby('gvkey', group_keys=False)[ffill_cols]
    .ffill()
)

In [ ]:
missing_two = char_cs_zscore_clean[fama_french_5_columns].isna().any(axis=1).mean()
missing_two

In [ ]:
for col in char_cs_zscore_clean.columns:
    num_missing = char_cs_zscore_clean[col].isna().mean()
    print(f"{col}: {num_missing:.4f} missing")

### Add Returns

### Writing

In [ ]:
#write results back to database
with engine.begin() as conn:
    char_cs_zscore.to_sql(
        "jkp_characteristics_cleaned_zscore",
        conn,
        if_exists="replace",
        index=False,
    )